<a href="https://colab.research.google.com/github/Peeyusj/gpt_from_sratch/blob/main/gpt_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 - download the dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-29 01:52:06--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.008s  

2026-04-29 01:52:07 (128 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# Cell 2 - read it and explore
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters:", len(text))
print("First 200 characters:")
print(text[:200])

Total characters: 1115394
First 200 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [3]:
vocab= sorted(set(text))
print(len(vocab))
print(vocab)

65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
stoi={}
itos={}
for i, ch in enumerate(vocab):
    itos[i]=ch
    stoi[ch]=i

print(stoi)
print(itos)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i',

In [5]:
def encode(strVal):
  res=[]
  for i in strVal:
    res.append(stoi[i])
  return res

def decode(intVal):
  res=[]
  for i in intVal:
    res.append(itos[i])
  return "".join(res)

print(encode("hello"))
print(decode(encode("hello")))


[46, 43, 50, 50, 53]
hello


In [6]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape)
print(data[:10])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [7]:
n = int(0.9 * len(data))

train_data =data[:n]   # everything before n
val_data =data[n:]   # everything from n onwards
print('train_data-',train_data.shape)
print('val_data-',val_data.shape)

train_data- torch.Size([1003854])
val_data- torch.Size([111540])


In [8]:
block_size = 8
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # now build x and y using ix
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [9]:
xb, yb = get_batch('train')
print(xb.shape)
print(yb.shape)
print("Input batch:")
print(xb)
print("\nTarget batch:")
print(yb)

torch.Size([4, 8])
torch.Size([4, 8])
Input batch:
tensor([[52, 43,  1, 44, 53, 56,  1, 58],
        [43,  1, 39,  0, 54, 43, 56, 44],
        [54, 39, 50, 39, 58, 43,  1, 39],
        [58, 46,  8,  1, 35, 46, 39, 58]])

Target batch:
tensor([[43,  1, 44, 53, 56,  1, 58, 46],
        [ 1, 39,  0, 54, 43, 56, 44, 43],
        [39, 50, 39, 58, 43,  1, 39, 42],
        [46,  8,  1, 35, 46, 39, 58,  1]])


In [10]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            return logits, None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            return logits, loss

    def generate(self, idx, max_new_tokens):
      for _ in range(max_new_tokens):
        logits, loss = self(idx)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
      return idx

In [11]:
vocab_size = 65
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8678, grad_fn=<NllLossBackward0>)


In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch('train')          # step 1: get batch
    logits, loss = model(xb, yb)         # step 2: forward pass
    optimizer.zero_grad()                # step 3: zero gradients
    loss.backward()                      # step 4: backward pass
    optimizer.step()                     # step 5: update weights

    if step % 1000 == 0:
        print(f"step {step}: loss {loss.item():.4f}")

step 0: loss 4.8563
step 1000: loss 4.1429
step 2000: loss 3.5134
step 3000: loss 3.3125
step 4000: loss 3.2134
step 5000: loss 2.7424
step 6000: loss 2.7307
step 7000: loss 2.4458
step 8000: loss 2.8188
step 9000: loss 2.6666


In [13]:
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=200)
print(decode(generated[0].tolist()))




TCEThethay,T:GLm aly;

TENCVYot s stn oul ouremight fon ne macomup e ane oy at be howit
IOO s tihe ath.'lf acot Prqug'd;&and bthzHyoe harou won. havaino wagt wou


Twitharen ge te.
'l thinyothe sin 


In [14]:
x = torch.tensor([
    [1.0, 2.0],   # token 0
    [3.0, 4.0],   # token 1
    [5.0, 6.0],   # token 2
    [7.0, 8.0],   # token 3
])

In [15]:
T, C = x.shape

x_avg = torch.zeros((T, C))

for t in range(T):
    x_prev = x[:t+1]
    x_avg[t] = x_prev.mean(dim=0)

print(x_avg)

tensor([[1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.]])


In [16]:
torch.tril(torch.ones(T, T))

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

In [17]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)
print(wei)

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])


In [18]:
x_avg2 = wei @ x
print(x_avg2)

tensor([[1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.]])


In [19]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
print(wei)

tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])


In [20]:
wei = F.softmax(wei, dim=-1)
print(wei)

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])


In [21]:
out = wei @ x
print(out)

tensor([[1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.]])


In [22]:
torch.manual_seed(1337)
head_size = 16

key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)

wei = q @ k.transpose(-2, -1)

print(k.shape)
print(q.shape)
print(wei.shape)
print(wei)

torch.Size([4, 16])
torch.Size([4, 16])
torch.Size([4, 4])
tensor([[ 2.0585,  7.2085, 12.3584, 17.5084],
        [ 5.2898, 17.6382, 29.9867, 42.3352],
        [ 8.5210, 28.0680, 47.6150, 67.1619],
        [11.7523, 38.4978, 65.2432, 91.9887]], grad_fn=<MmBackward0>)


In [23]:
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei)

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [4.3364e-06, 1.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0513e-17, 3.2424e-09, 1.0000e+00, 0.0000e+00],
        [1.4249e-35, 5.8774e-24, 2.4243e-12, 1.0000e+00]],
       grad_fn=<SoftmaxBackward0>)


In [24]:
wei = q @ k.transpose(-2, -1) * head_size**-0.5
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei)

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [4.3642e-02, 9.5636e-01, 0.0000e+00, 0.0000e+00],
        [5.6512e-05, 7.4890e-03, 9.9245e-01, 0.0000e+00],
        [1.9405e-09, 1.5551e-06, 1.2463e-03, 9.9875e-01]],
       grad_fn=<SoftmaxBackward0>)


In [25]:
v = value(x)      # transform each token through value layer
out = wei @ v     # weighted mix of value vectors
print(out.shape)

torch.Size([4, 16])
